In [ ]:
# ===========================================
# DEPRECATED
#
# Replacement:
# notebooks/02_silver/00_build_tournaments
# notebooks/02_silver/02_build_tournament_results
#
# Reason:
# Superseded by v2 schema. This notebook built
# tournament master + results together from
# event_result_raw and used the ephemeral
# event['id'] as tournament_id, colliding
# distinct tournaments. v2 splits tournament
# master (from tournament_list_raw, keyed by
# event_holding_id) from results (from
# event_result_raw) into separate notebooks.
# ===========================================


In [0]:
# ===========================================
# Notebook Name:
# 01_build_tournament_silver
#
# Purpose:
# Parse raw Pokemon tournament result JSON
# and build normalized Silver Delta tables.
#
# Input:
# pokemon.bronze.event_result_raw
#
# Output:
# pokemon.silver.tournaments
# pokemon.silver.tournament_results
# ===========================================

In [0]:
CATALOG_NAME = "pokemon"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

EVENT_RESULT_RAW_TABLE = (
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.event_result_raw"
)

TOURNAMENT_TABLE = (
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.tournaments"
)

TOURNAMENT_RESULT_TABLE = (
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.tournament_results"
)

print("Input :", EVENT_RESULT_RAW_TABLE)
print("Output:", TOURNAMENT_TABLE)
print("Output:", TOURNAMENT_RESULT_TABLE)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(EVENT_RESULT_RAW_TABLE)

display(
    bronze_df.select(
        "tournament_id",
        "http_status",
        "response_hash",
        "scraped_at",
        "ingestion_date",
        F.length("payload").alias("payload_length"),
    )
    .orderBy(F.col("scraped_at").desc())
)

In [0]:
latest_window = (
    Window
    .partitionBy("tournament_id")
    .orderBy(
        F.col("scraped_at").desc(),
        F.col("response_hash").desc(),
    )
)

latest_raw_df = (
    bronze_df
    .withColumn(
        "row_number",
        F.row_number().over(latest_window),
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

display(
    latest_raw_df.select(
        "tournament_id",
        "response_hash",
        "scraped_at",
    )
)

In [0]:
sample_row = (
    latest_raw_df
    .select("payload")
    .where(F.col("payload").isNotNull())
    .limit(1)
    .first()
)

if sample_row is None:
    raise ValueError(
        "event_result_raw に解析対象のpayloadがありません"
    )

sample_payload = sample_row["payload"]

print("Sample payload length:", len(sample_payload))
print(sample_payload[:500])

In [0]:
json_schema = (
    spark.range(1)
    .select(
        F.schema_of_json(
            F.lit(sample_payload)
        ).alias("json_schema")
    )
    .first()["json_schema"]
)

print(json_schema)

In [0]:
parsed_df = (
    latest_raw_df
    .withColumn(
        "parsed_payload",
        F.from_json(
            F.col("payload"),
            json_schema,
        ),
    )
    .select(
        "tournament_id",
        "source_url",
        "response_hash",
        "scraped_at",
        F.col("parsed_payload.code").alias("api_code"),
        F.col("parsed_payload.count").alias("result_count"),
        F.col("parsed_payload.event").alias("event"),
        F.col("parsed_payload.results").alias("results"),
    )
)

parsed_df.printSchema()
display(parsed_df)

In [0]:
tournaments_df = (
    parsed_df
    .withColumn(
        "event_json",
        F.to_json("event"),
    )
    .select(
        F.col("tournament_id"),
        F.get_json_object(
            "event_json",
            "$.event_title",
        ).alias("event_title"),
        F.get_json_object(
            "event_json",
            "$.eventTypeId",
        ).alias("event_type_id"),
        F.get_json_object(
            "event_json",
            "$.event_type_title",
        ).alias("event_type_title"),
        F.get_json_object(
            "event_json",
            "$.eventDaySupply",
        ).alias("event_date_display"),
        F.get_json_object(
            "event_json",
            "$.prefecture_name",
        ).alias("prefecture_name"),
        F.get_json_object(
            "event_json",
            "$.venue",
        ).alias("venue"),
        F.get_json_object(
            "event_json",
            "$.address",
        ).alias("address"),
        F.get_json_object(
            "event_json",
            "$.shopId",
        ).alias("shop_id"),
        F.get_json_object(
            "event_json",
            "$.shopName",
        ).alias("shop_name"),
        F.get_json_object(
            "event_json",
            "$.league",
        ).alias("league"),
        F.get_json_object(
            "event_json",
            "$.regulation",
        ).alias("regulation"),
        F.get_json_object(
            "event_json",
            "$.capacity",
        ).cast("int").alias("capacity"),
        F.col("result_count").cast("int"),
        F.col("source_url"),
        F.col("response_hash"),
        F.col("scraped_at").alias("source_scraped_at"),
        F.current_timestamp().alias("updated_at"),
    )
)

display(tournaments_df)

In [0]:
exploded_results_df = (
    parsed_df
    .select(
        "tournament_id",
        "source_url",
        "response_hash",
        "scraped_at",
        F.explode_outer("results").alias("result"),
    )
    .withColumn(
        "result_json",
        F.to_json("result"),
    )
)

display(exploded_results_df)

In [0]:
tournament_results_df = (
    exploded_results_df
    .select(
        F.col("tournament_id"),
        F.get_json_object(
            "result_json",
            "$.rank",
        ).cast("int").alias("rank"),
        F.get_json_object(
            "result_json",
            "$.player_id",
        ).alias("player_id"),
        F.get_json_object(
            "result_json",
            "$.name",
        ).alias("player_name"),
        F.get_json_object(
            "result_json",
            "$.area",
        ).alias("area"),
        F.get_json_object(
            "result_json",
            "$.point",
        ).cast("int").alias("point"),
        F.get_json_object(
            "result_json",
            "$.show_profile",
        ).cast("boolean").alias("show_profile"),
        F.get_json_object(
            "result_json",
            "$.deck_id",
        ).alias("deck_code"),
        F.col("source_url"),
        F.col("response_hash"),
        F.col("scraped_at").alias("source_scraped_at"),
        F.current_timestamp().alias("updated_at"),
    )
    .withColumn(
        "deck_url",
        F.when(
            F.col("deck_code").isNotNull(),
            F.concat(
                F.lit(
                    "https://www.pokemon-card.com/"
                    "deck/confirm.html/deckID/"
                ),
                F.col("deck_code"),
            ),
        ),
    )
    .withColumn(
        "deck_print_url",
        F.when(
            F.col("deck_code").isNotNull(),
            F.concat(
                F.lit(
                    "https://www.pokemon-card.com/"
                    "deck/print.html/deckID/"
                ),
                F.col("deck_code"),
                F.lit("/"),
            ),
        ),
    )
)

display(
    tournament_results_df.orderBy(
        "tournament_id",
        "rank",
    )
)

In [0]:
display(
    tournament_results_df
    .groupBy("tournament_id")
    .agg(
        F.count("*").alias("result_rows"),
        F.countDistinct("player_id").alias("unique_players"),
        F.countDistinct("deck_code").alias("unique_decks"),
        F.sum(
            F.when(
                F.col("deck_code").isNull(),
                1,
            ).otherwise(0)
        ).alias("missing_deck_count"),
        F.min("rank").alias("min_rank"),
        F.max("rank").alias("max_rank"),
    )
)

In [0]:
validation_df = (
    parsed_df
    .select(
        "tournament_id",
        F.col("result_count").cast("int").alias(
            "expected_count"
        ),
    )
    .join(
        tournament_results_df
        .groupBy("tournament_id")
        .agg(
            F.count("*").alias("actual_count")
        ),
        on="tournament_id",
        how="left",
    )
    .withColumn(
        "is_valid",
        F.col("expected_count") == F.col("actual_count"),
    )
)

display(validation_df)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TOURNAMENT_TABLE}
(
    tournament_id STRING NOT NULL,
    event_title STRING,
    event_type_id STRING,
    event_type_title STRING,
    event_date_display STRING,
    prefecture_name STRING,
    venue STRING,
    address STRING,
    shop_id STRING,
    shop_name STRING,
    league STRING,
    regulation STRING,
    capacity INT,
    result_count INT,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Normalized Pokemon tournament master data'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TOURNAMENT_RESULT_TABLE}
(
    tournament_id STRING NOT NULL,
    rank INT NOT NULL,
    player_id STRING,
    player_name STRING,
    area STRING,
    point INT,
    show_profile BOOLEAN,
    deck_code STRING,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    updated_at TIMESTAMP,
    deck_url STRING,
    deck_print_url STRING
)
USING DELTA
COMMENT 'Normalized Pokemon tournament ranking and deck results'
""")

In [0]:
tournaments_df.createOrReplaceTempView(
    "staged_tournaments"
)

spark.sql(f"""
MERGE INTO {TOURNAMENT_TABLE} AS target
USING staged_tournaments AS source
ON target.tournament_id = source.tournament_id

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *
""")

print("Tournament table merged.")

In [0]:
tournament_results_df.createOrReplaceTempView(
    "staged_tournament_results"
)

spark.sql(f"""
MERGE INTO {TOURNAMENT_RESULT_TABLE} AS target
USING staged_tournament_results AS source
ON  target.tournament_id = source.tournament_id
AND target.rank = source.rank

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *
""")

print("Tournament result table merged.")

In [0]:
display(
    spark.table(TOURNAMENT_TABLE)
    .orderBy(
        F.col("source_scraped_at").desc()
    )
)

In [0]:
display(
    spark.table(TOURNAMENT_RESULT_TABLE)
    .filter(
        F.col("tournament_id") == "1032136"
    )
    .orderBy("rank")
)

In [0]:
display(
    spark.sql(
        f"DESCRIBE HISTORY {TOURNAMENT_TABLE}"
    )
)

In [0]:
display(
    spark.sql(
        f"DESCRIBE HISTORY {TOURNAMENT_RESULT_TABLE}"
    )
)